In [ ]:
# 

In [ ]:
!pip install ~/fwiVis/utility_functions/
! pip install plotnine


In [ ]:
import s3fs
s3 = s3fs.S3FileSystem(anon=False)
from math import cos, asin, sqrt
import re

import numpy as np
import geopandas as gpd
import pandas as pd
from matplotlib import pyplot as plt
import os
import rioxarray as rio
import xarray as xr
import rasterio
import glob
from shapely.errors import ShapelyDeprecationWarning
from shapely.geometry import Point
import warnings
import folium
import datetime
import time
from folium import plugins
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 
#import contextily as cx
from shapely.geometry import box
import sys
from datetime import datetime, timedelta
from itertools import chain

from datetime import date
from bs4 import BeautifulSoup
import requests
import os
import plotnine
import xarray as xr

#import numpy as np
from matplotlib import pyplot as plt
from plotnine import ggplot, geom_point, geom_jitter, aes, stat_smooth, facet_wrap
import plotnine as plotnine
import seaborn as sns


In [ ]:
import fwiVis.fwiVis as fv

#path = "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_only/April_1_unmerged_fires_with_FWI.csv"
#path = "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/Final_dataset_as_of_20240209.csv"
#fire3 = fv.prep_fire_files(path)

path = os.path.abspath("data/Quebec_v3_full_data_perimeters20241112.csv")
fire3 = fv.prep_fire_files(path)
fire_first = fire3

#ciffc = pd.read_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/CIFFC_data/ciffc_all_canada.csv")
ciffc = pd.read_csv(os.path.abspath("contextual_data/CIFFC_data/ciffc_all_canada.csv"))
ciffc = ciffc[ciffc.field_agency_code == "qc"]

ciffc = gpd.GeoDataFrame(ciffc, geometry= gpd.points_from_xy(ciffc.field_longitude, ciffc.field_latitude), crs = "4326")
ciffc = ciffc.to_crs("3571")

ciffc["geometry_point"] = ciffc.geometry

In [ ]:
fire3 = fire3.sort_values(by = ["fireID", "t"])

fire3 = fire3[~fire3.FWI.isna()]
# df = fire3[fire3.fireID == '2367']
# df


In [ ]:

def corrected_farea_diff(df, var = "farea"):
    #sub_df = df[]
    df[f"{var}_diff"] = df[var].diff()
    min_t = df.t.min()
    if(np.isnan(*df.loc[df.t == min_t, [f"{var}_diff"]].values)):
        df.loc[df.t == min_t, [f"{var}_diff"]] = df.loc[df.t == min_t, [f"{var}"]].values
    else:
        print("Error! First value not NaN")
        df = None
    return(df)

def just_the_igs(df,): ## Reminder that this will only acocunt for the earliest ignition from a multi-fire complex. 

    df["is_ig"] = False
    df.loc[df.t == df.t.min(), ['is_ig']] = True
    
    return(df)

def extinction(df, days_past = 1):
    df.loc[:, ['is_ext']] = False
    df.loc[:, ["t"]] = df.t.astype("datetime64[ns]")
    final_t = df[df.n_newpixels > 0].t.astype("datetime64[ns]").max()
    post_fire = final_t + timedelta(days = days_past)

    if (pd.isnull(np.datetime64(str(final_t)))):
        df.loc[(df.t == df.t.min()), ['is_ext']] = True
    else:
        df.loc[(df.t  > final_t)  & (df.t  <= post_fire), ['is_ext']] = True
    #print(final_t)
    #print(df.fireID.unique())
    #print(len(df[df["is_ext"] == True]))
    return(df)


fire3 = fire3[~fire3.FWI.isna()]

post_fr = fire3

post_fr = post_fr.groupby("fireID").apply(extinction).reset_index(drop = True)
post_fr = post_fr.groupby("fireID").apply(just_the_igs).reset_index(drop = True)

fire3 = fire3.groupby("fireID").apply(corrected_farea_diff).reset_index(drop = True)
fire3 = fire3.groupby("fireID").apply(just_the_igs).reset_index(drop = True)
#fire3.farea = fire3.farea.astype("int64")
fire3 = fire3.sjoin(ciffc)
fire3["farea_diff"] = fire3.groupby("fireID").farea.diff()

row_mask = (~fire3.fireID.str.contains("_"))

#fire3[row_mask].hvplot.scatter(x='FWI', y='farea_diff', hover_cols=['fireID', 't'])
#fire3[row_mask].plot.scatter(x='FWI', y='farea_diff')

In [ ]:
supression = fire3.groupby("fireID").field_response_type.unique().reset_index()
supression

supression["len"] = supression.field_response_type.apply(len)

multi_suppress_ids = supression[supression.len> 1].fireID

supression[supression.len> 1]
fire_ids_not_unique_supression = supression[supression.len> 1].fireID

In [ ]:
### Function that takes a point and looks for the multi polygons that intersect with that point. 
#It then assignes just the polygon portion of the multipolygon with the point's ID. 
# At the end, I want to put in an ID where there are two supression stratagies, and I want to get two IDs out, one per supression stratagy: subset to just the portion where there is one supression strategy.  
# I have found one fire so far (1082) where this wont work, because at least some probable fire that could/should be tied to one surpession strategy doesn't overlap with the point, even after the reported time, and by the time out polygon overlapped with the point it was already merged with anouther one. So boo. 

import shapely
import shapely.geometry as gmt
from shapely.geometry import MultiPolygon, Point, Polygon

def split_multipolygon(multipolygon, point):
    """Splits a MultiPolygon into separate polygons based on intersection with a point.

    Args:
        multipolygon (MultiPolygon): The MultiPolygon to split.
        point (Point): The point used to determine intersection.

    Returns:
        list: A list of individual polygons.
    """

    polygons = []
    for polygon in multipolygon:
        if polygon.intersects(point):
            polygons.append(polygon)
    return polygons


def exactly_one_true(lst):
    """Checks if exactly one element in the list is True."""
    return sum(lst) == 1

def sep_supression(fid, df1,  meter_crs = 3571, point_crs = 4326): #df2,
    print("Assuming df1 is in  crs 4326, and df2 is in a metered crs (35")
    df = df1[df1.fireID == fid]
    #df2 = df2[df2.fireID == fid]
    #df = df[30:40]
    # df = df.to_crs(meter_crs)
    # df.geometry = df.geometry.buffer(500) ## 200 meeters
    # df = df.to_crs(point_crs)
    ciffc_resp = df.field_response_type.unique()
    resp_l = []
    #for r in ciffc_resp:
    just_intersect = []
    #unique_points = df[["field_latitude", "field_longitude", "field_response_type", "field_agency_fire_id"]].drop_duplicates().reset_index(drop = True) # (df.field_response_type == r)
    #unique_points = gpd.GeoDataFrame(unique_points, geometry= gpd.points_from_xy(unique_points.field_longitude, unique_points.field_latitude,  crs = 4326))

    unique_points = df[["geometry_point", "field_response_type", "field_agency_fire_id"]].drop_duplicates().reset_index(drop = True)
    unique_points = gpd.GeoDataFrame(unique_points, geometry= unique_points.geometry_point,  crs = 3571)
    #return(unique_points)
    #points = [Point(lon, lat) for lon, lat in zip(unique_points.field_longitude, unique_points.field_latitude)]
    points = unique_points.geometry_point
    #print(len(points))
    #return(points)
    
    for index, row in df.iterrows(): # [df.field_response_type == r]
    #if isinstance(row['geometry'], MultiPolygon):
        #print("This is a multipolygon")
       #split_polygons = split_multipolygon(row['geometry'], intersection_point)
        ## Double check that only 1 point intersects with polygon. 
        if(isinstance(row['geometry'], Polygon)):
            row['geometry'] = gmt.MultiPolygon([row['geometry']])
        for polygon in row['geometry'].geoms:
            #print(shapely.intersects(points, polygon))
            unique_points["Does_it_intersect"] = np.nan
            intersecting_points = []
            for point in points:
                intersecting_points.append(polygon.intersects(point))
                    
            unique_points["Does_it_intersect"] = intersecting_points
            up_group = unique_points.groupby("field_response_type").Does_it_intersect.unique().reset_index()
            #return(up_group)

            only_one_type_of_supression_intersects = (sum(up_group.Does_it_intersect.explode().values) == 1)
            #intersecting_points = [point for point in points if polygon.intersects(point)]
            #print(intersecting_points)

            #if len(intersecting_points) == 1:  # Exactly one point intersects
            if only_one_type_of_supression_intersects: ### Only one catagory intersects with this polygon
                #print(only_one_type_of_supression_intersects)
                #return(up_group)
                new_row = row.copy()
                new_row["geometry"] = polygon
                new_row["farea"] = (polygon.area / (1000 * 1000))
                new_row["fperim"] = np.nan
                new_row["meanFRP"] = np.nan
                
                
                
                # try:
                    #new_row["fireID"] = str(fid) + "." +str(up_group[up_group.Does_it_intersect.sum()].field_response_type.iloc[0])# tmp2[tmp2.source.explode()].degree.iloc[0]
                    #new_row["fireID"] = str(fid) + "." +str(up_group[up_group.Does_it_intersect.apply(any)].field_response_type.iloc[0]) + "." + str(*unique_points[unique_points.Does_it_intersect == True].field_agency_fire_id.unique())
                sup = str(up_group[up_group.Does_it_intersect.apply(any)].field_response_type.iloc[0])
                new_row["fireID"] = str(fid) + "." + sup +"." +  ".".join(unique_points[unique_points.Does_it_intersect == True].field_agency_fire_id.astype("str"))
                new_row["field_response_type"] = sup
                    
                # except:
                #     return(unique_points)
                # #     return(up_group)
                    
                just_intersect.append(new_row)
                # fig, ax = plt.subplots(figsize=(8, 6))
                # #print(type(new_row))
                # #new_row.plot(color = "green")
                # plt.plot(*new_row["geometry"].exterior.xy, color = "green")
                # for point in points:
                #     ax.plot(point.x, point.y, 'ro', label="Point")  # 'ro' for red points
                
                #     # Adjust the plot
                #     ax.set_title("Geometry and Points")
                #     ax.set_xlabel("Longitude")
                #     ax.set_ylabel("Latitude")
                #     plt.grid(True)
                    
                #     # Show the plot
                #     plt.show()
                #plt.plot(*new_row["geometry"].exterior.xy, color = "green")
                #plt.plot(*unique_points["geometry"].exterior.xy, color = "red")
                #unique_points.plot(color = "red")
                #plt.plot(unique_points.field_longitude, unique_points.field_latitude,  color = "red")
                #plt.show()
            else:
                new_row = row.copy()
                new_row["geometry"] = polygon
                # fig, ax = plt.subplots(figsize=(8, 6))
                # plt.plot(*new_row["geometry"].exterior.xy, color = "yellow")
                # for point in points:
                #     ax.plot(point.x, point.y, 'ro', label="Point")  # 'ro' for red points
                
                #     # Adjust the plot
                #     ax.set_title("Geometry and Points")
                #     ax.set_xlabel("Longitude")
                #     ax.set_ylabel("Latitude")
                #     plt.grid(True)
                    
                #     # Show the plot
                #     plt.show()
                #plt.plot(*new_row["geometry"].exterior.xy, color = "yellow")
                #plt.plot(unique_points.field_longitude, unique_points.field_latitude,  color = "red")
                #plt.plot(*unique_points["geometry"].exterior.xy, color = "red")
                #unique_points.plot(color = "red")
                #plt.show()
                #print("skipping")
                #just_intersect.append(None)
    # else:
    #     just_intersect.append(row)
    just_intersect_df = gpd.GeoDataFrame(just_intersect)
    just_intersect_df = gpd.GeoDataFrame(just_intersect_df, geometry = just_intersect_df.geometry, crs = 3571)
    #just_intersect_df = just_intersect_df[just_intersect_df.field_response_type == sup] # Dropping false lable
    just_intersect_df = just_intersect_df.drop_duplicates()
    # if(len(just_intersect_df) > 0):
    #     #print(just_intersect)
    #     just_intersect_df.fireID = just_intersect_df.fireID.astype("str") + "." + r
    #     #print(just_intersect_df)
    #     resp_l.append(just_intersect_df)
    # else:
    #     print(f"{fid} had no independant multi-polygons to split")
#full_df = pd.concat(resp_l, axis=0)
    return(just_intersect_df)
    

def check_that_continious_record(df):
    min_t = df.t.min()
    max_t = df.t.max()
    #dates = pd.date_range(start= min_t, end=  max_t).to_pydatetime().tolist()
    dates = pd.date_range(start= min_t, end=  max_t).strftime('%Y-%m-%d 12:00:00').tolist()
    len_df = len(df.t.unique())
    #print(df.t.unique())
    len_seq = len(dates)
    #print(dates)
    if(len_df != len_seq):
        print(f"FireID {df.fireID.unique()} is not a continious sequence.")
        return(False)
    d_list = []
    for d in dates:
        some_dates = df[df.t == d]
        d_list.append(len(some_dates) > 0)
    return(any(d_list))

In [ ]:
### remake IDs with duel supression status 


skip_list = ["1082", 
            "1324"] ### has more than one supression, but seems like the reported point doesn't overlap with an indepandant multipolygon. 

duel_sup =  [item for item in fire_ids_not_unique_supression if item not in skip_list]
dfs = []
for s in duel_sup:
    print(s)
    tmp = sep_supression(str(s), fire3)
    dfs.append(tmp)
foo = pd.concat(dfs)

foo = gpd.GeoDataFrame(foo, geometry = foo.geometry, crs = 3571)
#foo = foo.set_crs(4326)


#### Check that there is a continious t 


bools = foo.groupby("fireID").apply(check_that_continious_record)
print(f"Are all the fires continuios records???: {all(bools)}")

if(all(bools)):
    fire3 = fire3[~fire3.fireID.isin(fire_ids_not_unique_supression)]
    foo = foo.to_crs(fire3.crs)
    fire3 = pd.concat([fire3, foo], ignore_index=True)

In [ ]:
def chop_fires_at_end(df):
    final_t = df[df.n_newpixels > 0].t.max()
    df = df[df.t <= final_t]
    return(df)

#post_fr = fire3.groupby("fireID").apply(extinction).reset_index(drop = True)
fire3 = fire3.groupby("fireID").apply(chop_fires_at_end).reset_index(drop = True)


fire3 = fire3.sort_values(by = ["fireID", "t"])
fire3 = fire3[~fire3.FWI.isna()]

In [ ]:
#fire3["farea_diff"] = fire3.groupby("fireID").farea.diff()
#fire3 = fire3.groupby("fireID").apply(corrected_farea_diff)
fire3["FWI_rolling"] = fire3.groupby("fireID").FWI.rolling(3).max().reset_index(drop = True)
fire3["FWI_diff"] = fire3.groupby("fireID").FWI.diff()

fire3["farea_diff_stand"] = fire3.groupby("fireID").farea.diff()



## Fireline 
fire3["flinelen_shifted"] = fire3.groupby("fireID").flinelen.shift(periods = 1)
fire3["fperim_shifted"] = fire3.groupby("fireID").fperim.shift(periods = 1)
fire3["flinelen_diff"] = fire3.groupby("fireID").flinelen.diff()
fire3["fperim_diff"] = fire3.groupby("fireID").fperim.diff()


### Some useful vars to color by 
def assign_day_of_fire(df):
    df = df.sort_values(by = "t")
    df['day_of_fire'] = df.t.rank()
    #df['day_of_fire'] = df['day_of_fire'].astype("int64")
    return(df)


def get_max_duration(df):
    max_duration = df.duration.max()
    df["max_duration"] = max_duration


# fire3 = fire3.groupby("fireID").apply(assign_day_of_fire).reset_index(drop = True)
# fire3["farea_shifted"] = fire3.groupby("fireID").farea.shift(periods = 1)

# fire3["normalized_farea_diff"] = fire3.farea_diff/fire3.farea_shifted



fire3["GEOS_IMERGEARLY"] = fire3["GEOS-5.IMERGEARLY"]

def line_trend(df, var):
    df = df.sort_values(by = "t")
    df["time_delta"] = df.t.astype("datetime64[ns]") - df.t.astype("datetime64[ns]").min()
    
    df["t_float"] = df["time_delta"].dt.days
    z = np.polyfit(df.t_float, df[var], 1)
    df["trend"] = z[0] ## Slope of trend line
    return(df)


In [ ]:

### Implementing FBP technical handbook equation 88 https://drive.google.com/file/d/1xBseasON22KFC4jEVBKk75EOuUK-dpM2/view

#1) Calculate the length and breadth of the fires. Simple way, get just the convex hull and the longer one is length. More complex, length shoudl maybe be tied to where active fire line is (to represent head fire?)
from shapely.geometry import LineString

def get_length_to_breadth(polygon):
    
    mbr_points = list(zip(*polygon.minimum_rotated_rectangle.exterior.coords.xy)) ### oh no projection issues? 
    # calculate the length of each side of the minimum bounding rectangle
    mbr_lengths = [LineString((mbr_points[i], mbr_points[i+1])).length for i in range(len(mbr_points) - 1)]

    # get major/minor axis measurements
    minor_axis = min(mbr_lengths)
    major_axis = max(mbr_lengths)
    
    return(major_axis/minor_axis)



fire3["LB"] = fire3.geometry.apply(get_length_to_breadth)


## Calculate spread rate

row_mask = (fire3.fperim_diff > 0.1) & (fire3.flinelen > 0)
perimeter_growth_rate = fire3[row_mask]["fperim_diff"] ### Modify by length of active fire line? 
LB = fire3[row_mask]["LB"]

def get_perimeter_growth_rate(df):
    term_a = ((1/df["LB"]) + 1) # will this vectorize? 
    term_b =  1 + (((df["LB"] - 1) / (2 * (df["LB"] + 1)))**2)

    df["tot_ros_all_perim"] = 2 * (df["fperim_diff"] / (term_a * term_b)) ### Will have same units as perimeter growth in m/ 24 hour. Not area. 
    df["tot_ros_flinelen_shifted"] = 2 * (df["flinelen_shifted"] / (term_a * term_b)) ### Will have same units as perimeter growth in m/ 24 hour. Not area. 
    return(df)



fire3 = fire3.groupby("fireID").apply(get_perimeter_growth_rate).reset_index(drop = True)

# fire3["spread_rate"] = np.nan
# fire3.loc[row_mask, "spread_rate"] = 1/(2*((np.pi * (1+(1/LB)) * (1 + ((LB -1)/(2*(LB+1)))**2)) / perimeter_growth_rate ))


# fire3["flinelen_rolling"]  = fire3.groupby("fireID").flinelen.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
# fire3["flinelen_diff_rolling"] = fire3.groupby("fireID").flinelen_diff.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
# fire3["fperim_rolling"] = fire3.groupby("fireID").fperim.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
# fire3["fperim_diff_rolling"] = fire3.groupby("fireID").fperim_diff.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
# fire3["spread_rate_rolling"] = fire3.groupby("fireID").spread_rate.rolling(rolling_num).agg(agg_function).reset_index(drop = True)


# long_fires = fire3[fire3.day_of_fire > 3].fireID.unique()
# #fire3['max_dof'] = fire3.groupby("fireID").day_of_fire.max()

# row_mask = (~fire3.fireID.str.contains("_")) & (fire3.farea_diff > 0.1) & (fire3.fireID.isin(long_fires)) #& fire3['max_dof'] >= 3#& (fire3.max_duration >= 3)

In [ ]:
y_vals = ["farea_diff", "farea_diff_stand", "flinelen", "flinelen_diff", 'tot_ros_all_perim', 'tot_ros_flinelen_shifted']
rolling_nums = [3, 7]
agg_functions = ["mean", "max"] ## Need to do line_trend seperately. 
#agg_function = "max" # max

x_rolling = ["GEOS_IMERGEARLY", "FWI"]
x_rolling2 = x_rolling.copy()
y_vals1 = y_vals.copy()
for rolling_num in rolling_nums:
    
    for xr in x_rolling:
        # var_name_tmp = f"{xr}_line_trend"
        # fire3[var_name_tmp] = fire3.groupby("fireID").rolling(rolling_num).agg(line_trend, var = xr).reset_index(drop = True)
        # x_rolling2.append(var_name_tmp)
        for agg_function in agg_functions:
            var_name_tmp = f"{xr}_rolling_{rolling_num}_{agg_function}"
            fire3.loc[:, [var_name_tmp]] = fire3.groupby("fireID")[xr].rolling(rolling_num).agg(agg_function).reset_index(drop = True)
            x_rolling2.append(var_name_tmp)
    for yv in y_vals:
        for agg_function in agg_functions:
            var_name_tmp = f"{yv}_rolling_{rolling_num}_{agg_function}"
            fire3.loc[:, [var_name_tmp]] = fire3.groupby("fireID")[yv].rolling(rolling_num).agg(agg_function).reset_index(drop = True)
            y_vals1.append(var_name_tmp)

# fire3["FWI_rolling"] = fire3.groupby("fireID").FWI.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
# fire3["FWI_diff_rolling"] = fire3.groupby("fireID").FWI_diff.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
# fire3['farea_rolling'] = fire3.groupby("fireID").farea.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
# fire3['farea_diff_rolling'] = fire3.groupby("fireID").farea_diff.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
# fire3["GEOS-5.IMERGEARLY_rolling"] = fire3.groupby("fireID")["GEOS-5.IMERGEARLY"].rolling(rolling_num).agg(agg_function).reset_index(drop = True)

#long_fires = fire3[fire3.day_of_fire > 3].fireID.unique()
#fire3['max_dof'] = fire3.groupby("fireID").day_of_fire.max()

#row_mask = (~fire3.fireID.str.contains("_")) & (fire3.farea_diff > 0.1) & (fire3.fireID.isin(long_fires)) #& fire3['max_dof'] >= 3#& (fire3.max_duration >= 3)

In [ ]:
# row_mask = (fire3.fperim_diff > 0.1)

# plt.scatter( fire3[row_mask]['GEOS-5.IMERGEARLY'], fire3[row_mask].tot_ros_all_perim)

In [ ]:
# row_mask = (fire3.fperim_diff > 0.1)

# plt.scatter( fire3[row_mask]['GEOS-5.IMERGEARLY'], np.log(fire3[row_mask].tot_ros_flinelen_shifted))
# plt.show()

# plt.scatter( fire3[row_mask]['GEOS-5.IMERGEARLY'], fire3[row_mask].tot_ros_flinelen_shifted)
# plt.show()

In [ ]:
# plt.scatter( fire3.farea, fire3.farea_diff)
# plt.show()

In [ ]:
# plt.scatter( fire3.farea, fire3.flinelen)
# plt.show()

In [ ]:
# plt.scatter( x = fire3.farea_diff, y = fire3.flinelen)
# plt.show()

In [ ]:
# row_mask = (fire3.farea_diff > 0.4)

# from math import sqrt
# def mod_list(old):
#     return [x/(1.5) for x in old]

# seq = list(range( 0, round(fire3[row_mask].farea_diff.max()), round(fire3[row_mask].farea_diff.max()/ len(fire3[row_mask].farea_diff))))
# plt.scatter( x = fire3[row_mask].farea_diff, y = fire3[row_mask].flinelen)
# plt.plot(seq, mod_list(seq))
# plt.xlabel("Fire Area Growth Km^2")
# plt.ylabel("Active Fire Line Km")
# plt.show()

In [ ]:

# seq = list(range( 0, round(fire3[row_mask].farea_diff.max()), round(fire3[row_mask].farea_diff.max()/ len(fire3[row_mask].farea_diff))))
# plt.scatter( x = fire3[row_mask].farea_diff, y = fire3[row_mask].flinelen_shifted)
# plt.plot(seq, mod_list(seq))
# plt.xlabel("Fire Area Growth Km^2")
# plt.ylabel("Active Fire Line of previous fire")
# plt.show()

In [ ]:
# ### Check how the perimeter transforms into area

# plt.scatter( x = fire3[row_mask].farea_diff, y = fire3[row_mask].flinelen_shifted)
# plt.plot(seq, mod_list(seq))
# plt.xlabel("Fire Area Growth Km^2")
# plt.ylabel("Active Fire Line of previous fire")
# plt.show()

In [ ]:



# plt.scatter( x = fire3['GEOS-5.IMERGEARLY'], y = fire3.flinelen_shifted/fire3.fperim_shifted)
# plt.xlabel("FWI")
# plt.ylabel("Active Fire Line of Previous Day")
# plt.show()

In [ ]:
# plt.scatter( x = fire3['GEOS-5.IMERGEARLY'], y = fire3.flinelen)
# plt.xlabel("Fire Area Growth Km^2")
# plt.ylabel("Active Fire Line of previous fire")
# plt.show()

In [ ]:

# plt.scatter( x = fire3['GEOS-5.IMERGEARLY'] * (fire3.flinelen_shifted/fire3.fperim_shifted), y = fire3.farea_diff)
# plt.xlabel("FWI * active fire line")
# plt.ylabel("Fire Growth")
# plt.show()

In [ ]:
# plt.scatter( x = fire3[fire3.farea_diff > 0.1]['GEOS-5.IMERGEARLY'], y = (fire3[fire3.farea_diff > 0.1].farea_diff) ** (1/2))
# plt.xlabel("FWI * active fire line")
# plt.ylabel("Fire Growth")
# plt.show()

In [ ]:
# plt.scatter( x = fire3['GEOS-5.IMERGEARLY'], y = fire3.farea_diff_stand)
# plt.xlabel("FWI * active fire line")
# plt.ylabel("Fire Growth")
# plt.show()

In [ ]:
### Looking at just the post-fire vs ignition FWI

colors = ['b','r']
labels = ["FWI After Fire Stopped", "FWI When Fire Started"]
alphas = [0.3, 0.3]

fig, ax1 = plt.subplots()
ax1.hist([post_fr[post_fr.is_ext == True]['GEOS-5.IMERGEARLY'],post_fr[post_fr.is_ig == True]['GEOS-5.IMERGEARLY']],color=colors, label = labels)
#ax1.set_xlim(-10,10)
ax1.set_ylabel("Count")
plt.tight_layout()
plt.show()

print(f"N of {len(post_fr[post_fr.is_ext == True]['GEOS-5.IMERGEARLY'])} from extinquishing fires.")
print(f"N of {len(post_fr[post_fr.is_ig == True]['GEOS-5.IMERGEARLY'])} from ignighting fires.")

In [ ]:
post_fr[post_fr.is_ext == True]['GEOS-5.IMERGEARLY'].hist(color = "blue", alpha = 0.5, label = "FWI After Fire Stopped")
post_fr[post_fr.is_ig == True]['GEOS-5.IMERGEARLY'].hist(figsize=(8, 4), color = "red", alpha = 0.5, label = "FWI When Fire Started")

print(f"N of {len(post_fr[post_fr.is_ext == True])} from extinquishing fires.")
print(f"N of {len(post_fr[post_fr.is_ig == True])} from ignighting fires.")
plt.legend()
plt.title("FWI Values Including IMERGE")

# Display the plot    
plt.savefig(os.path.abspath("exploratory_figs/igs_vs_ext_hist_geos5.png"))
plt.show()

In [ ]:
  
post_fr[post_fr.is_ext == True]['FWI'].hist(color = "blue", alpha = 0.5, label = "FWI After Fire Stopped")
post_fr[post_fr.is_ig == True]['FWI'].hist(figsize=(8, 4), color = "red", alpha = 0.5, label = "FWI When Fire Started")
plt.legend()
plt.title("FWI Values GEOS-5 Only")
# Display the plot    
plt.savefig(os.path.abspath("exploratory_figs/igs_vs_ext_hist_IMERGE.png"))
plt.show()

## Data prepereation steps
- chop of fires at end
- create all varaibles
- merge with CIFFC



## Aspects of fires to relate to FWI

- farea_diff_stand (standard)
- farea_diff (corrected - first observation is included)
- flinelen - length of the active fireline
- eliisoid equations based on perimeter growth
- ellipsoid equations based on active fire line
- active fireline as a fracton of perimeter growth


## Data translformations to fire aspects

- square-root
- lagged
- logged
- square-root + lagged
- logged + lagged
- rolling (?) -- leave off?
   - diff rolling functions
   - mean
   - max and then drop non-uqnique vals
   - mean of previous 3 days
   - mean (7 days)
   - trend from last 3 (4?) days (increasing or decreasing)


## FWI data transforms

- anticedent conditions mean ------ mean rolling?
- mean of X days before a given spread
- logit model of grow/not grow FWI


## How test? 
- heteroskedasticity-friendly metric of performence (Spearmans's)
- Run them all, in a tiered way.
- Include figures that show all the corrections. / transformations. 
- 
- 

In [ ]:
from scipy import stats
from patsy import ModelDesc
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

def joint_plot(fire_delta, x, y):
        # plt.scatter(small_fire.FWI, (small_fire[f"fwi_error_{i}"] * small_fire[f"fwi_error_{i}"])**(1/2))
    #fire_delta[y] = fire_delta[y].astype("int")
    #fire_delta = fire_delta[[x, y]].dropna()
    fire_delta = fire_delta[[x, y]].drop_duplicates()

    formula = f"{y} ~ {x}"
    #links = "Identity"
    links = "Log"
    #families = "NegativeBinomial"
    
    families = "Gaussian"
    link_g = getattr(sm.genmod.families.links, links)
    method = getattr(sm.families, families)
    fire_delta = fire_delta[[x , y]].dropna()
    model = smf.glm(formula, data=fire_delta, family=method(link = link_g(), check_link=True)).fit()
    
    sns.jointplot(data=fire_delta, x= x, y=y, marker="+",  dropna = True, marginal_kws = {"bins": 20}) #  marginal_kws = {"log_scale": True} 
    #t.ax_marg_y.set_yscale("log")
    max_of_max = round(max(fire_delta[x].max(), fire_delta[y].max()))
    min_of_min = round(min(fire_delta[x].min(), fire_delta[y].min()))
    fire_delta['fitted'] = model.fittedvalues
    fire_delta['residuals'] = model.resid_response
    fire_delta = fire_delta.sort_values(by = x)
    predictions = model.get_prediction(fire_delta, transform = True) #df, transform = False
    fire_delta['predicted'] = predictions.predicted_mean
    fire_delta['conf_int_low'], fire_delta['conf_int_high'] = predictions.conf_int().T
    plt.fill_between(fire_delta[x], fire_delta['conf_int_low'], fire_delta['conf_int_high'], alpha=0.2, color = "yellow")
    


    sns.lineplot(x=x, y='predicted', data=fire_delta, legend = False)
    #plt.plot(range(min_of_min, max_of_max), range(min_of_min, max_of_max), color = "black")
    #plt.xlabel(x)
    #plt.ylabel(y)
    
    ## Stats
    r_sq = 1 - (np.sum((fire_delta['residuals'])**(2)) / np.sum((fire_delta[y] - fire_delta[y].mean())**(2)))
    #bias = np.sum(fire_delta[y] - fire_delta[x] )/len(fire_delta[x])
    res = stats.spearmanr(fire_delta[x], fire_delta[y], nan_policy = 'omit')
    spear = res.statistic
    p_val_spear = res.pvalue
    plt.title("")
    #print("R^2 from modeled line: " + str(r_sq) )
    #print("Spearman Correlation: " + str(spear) )
    plt.savefig(f'{os.path.abspath("exploratory_figs")}/{y}_vs_{x}.png')
    #plt.show()
    plt.close()
    return([x, y, r_sq, spear, p_val_spear])

In [ ]:
problem_vars = ['tot_ros_all_perim',  'tot_ros_flinelen_shifted']
lags = [-2, -1, 1, 2, 3]

for p in problem_vars:
    fire3[ fire3[p] < 0][p] = 0


y_vals2 = y_vals1.copy()

In [ ]:
for v in y_vals:
    for i, l in enumerate(lags):
        lag_name = l
        if(l < 0):
            lag_name = "minus_" + str(np.absolute(l))
        var_name = f"{v}_shifted_{lag_name}"
        fire3[var_name] = fire3.groupby("fireID")[f"{v}"].shift(periods = l).reset_index(drop = True)
        y_vals2.append(var_name)

In [ ]:
 





df_stats = {
    "x":[None], 
    "y": [None],
    "r_sq": [None],
    "spear": [None], 
    "pval": [None]
}
stats_sum = pd.DataFrame(df_stats)
x = 'GEOS_IMERGEARLY'

combinations = [(x, y) for x in x_rolling2 for y in y_vals2]


for i, c in enumerate(combinations):
    x = c[0]
    y = c[1]
    tmp = joint_plot(fire3, x,  y)
    stats_sum.loc[i, "x"] = tmp[0]
    stats_sum.loc[i, "y"] = tmp[1]
    stats_sum.loc[i, "r_sq"] = tmp[2]
    stats_sum.loc[i, "spear"] = tmp[3]
    stats_sum.loc[i, "pval"] = tmp[4]

In [ ]:
stats_sum = stats_sum.sort_values(by = "spear", ascending= False)
print(f"The largest spearman correlation is between {} and: {stats_sum.y.iloc[0]} at {stats_sum.spear.iloc[0]}, {stats_sum.y.iloc[1]} at {stats_sum.spear.iloc[1]}, and {stats_sum.y.iloc[2]} at {stats_sum.spear.iloc[2]}")




In [ ]:
stats_sum = stats_sum.sort_values(by = "r_sq", ascending= False)
print(f"The largest R^2 (from modeled line) is between fwi and: {stats_sum.y.iloc[0]} at {stats_sum.r_sq.iloc[0]}, {stats_sum.y.iloc[1]} at {stats_sum.r_sq.iloc[1]}, and {stats_sum.y.iloc[2]} at {stats_sum.r_sq.iloc[2]}")